# Manual Build + Validate (no LLM)

This notebook lets you provide a custom `docker_build_pkg.sh` script (building_data) and validate it end-to-end against a repo commit using the datasmith Docker pipeline.

Flow:
- Edit the parameters below (owner/repo/sha, optional python version, output paths).
- Paste your build script into `building_data`.
- Run the Build + Validate cell.
- On success, the context is registered and saved to the chosen context registry JSON.

In [1]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
from __future__ import annotations

import sys
import uuid
from pathlib import Path

# Ensure 'src' is importable (run this notebook from repo root)
SRC = (Path.cwd() / 'src').resolve()
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

import argparse
import json

import docker
import pandas as pd

from datasmith.agents.build import _handle_success, build_once_with_context
from datasmith.core.models import Task
from datasmith.docker.cleanup import remove_containers_by_label
from datasmith.docker.context import ContextRegistry, DockerContext
from datasmith.docker.orchestrator import gen_run_labels
from datasmith.docker.validation import DockerValidator, ValidationConfig
from datasmith.logging_config import configure_logging

configure_logging(level=20)

print('Imports OK')

/mnt/sdd1/atharvas/formulacode/datasmith


04:27:55 WARNING  simple_useragent.core: Falling back to historic user agent.


Imports OK


## Parameters

Fill these in for your repo/commit, context-registry file, and output directory.


In [2]:
COMMITS_PATH = Path('scratch/artifacts/pipeflush/merge_commits_filtered_with_patch.parquet')
CONTEXT_REGISTRY_PATH = Path('scratch/merged_context_registry_2025-10-08T09:22:23.904388.json')
OUTPUT_DIR = Path('scratch/artifacts/pipeflush/results_synthesis_manual/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Prepare Docker + Registry

In [3]:
client = docker.from_env()
# Load or create a context registry
if CONTEXT_REGISTRY_PATH.exists():
    registry = ContextRegistry.load_from_file(CONTEXT_REGISTRY_PATH)
    print('Loaded registry from', CONTEXT_REGISTRY_PATH)
else:
    registry = ContextRegistry()
    print('Created new registry')

# Validation config (adjust timeouts if needed)
config = ValidationConfig(
    output_dir=OUTPUT_DIR,
    build_timeout=45 * 60,
    run_timeout=20 * 60,
    tail_chars=10_000,
)
validator = DockerValidator(client=client, context_registry=registry, machine_defaults={}, config=config)
print('Docker + registry ready')

Loaded registry from scratch/merged_context_registry_2025-10-08T09:22:23.904388.json
Docker + registry ready


In [4]:
commits_df = pd.read_parquet(COMMITS_PATH)
commits_df

,sha,date,message,total_additions,total_deletions,total_files_changed,files_changed,original_patch,has_asv,file_change_summary,...,pr_base_trees_url,pr_base_updated_at,pr_base_url,pr_base_visibility,pr_base_watchers,pr_base_watchers_count,pr_base_web_commit_signoff_required,pr_base_sha,container_name,patch
0,01fbbe37b2754f056b5241deef5f987482dc897e,2020-07-09T21:55:34+02:00,Memory leak testing using valgrind (#159)\n\n,49,0,3,.dockerignore\nREADME.rst\ndocker/Dockerfile.v...,From 01fbbe37b2754f056b5241deef5f987482dc897e ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,d51e87ec1bd230bffb05882b3bf84b52540a89d1,pygeos-pygeos-d51e87ec1bd230bffb05882b3bf84b52...,diff --git a/.dockerignore b/.dockerignore\nne...
1,c05704f69fd0b9ec57fe4b31833736c156eab5b4,2021-07-05T12:57:17+02:00,BUG: fix no inplace output check for box and s...,4,4,1,src/ufuncs.c,From c05704f69fd0b9ec57fe4b31833736c156eab5b4 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,cec21ae5f11d9038a7d6cb5ba4fa6f4424eb9b3c,pygeos-pygeos-cec21ae5f11d9038a7d6cb5ba4fa6f44...,diff --git a/src/ufuncs.c b/src/ufuncs.c\ninde...
2,65203a9e763019c42865be1423e267f94ca3f649,2020-11-29T16:03:51-08:00,ENH: Adds reverse function for GEOS >= 3.7 (#2...,136,11,4,CHANGELOG.rst\npygeos/constructive.py\npygeos/...,From 65203a9e763019c42865be1423e267f94ca3f649 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,3485ffb0db54f055a5379946dcc31984fb8b8853,pygeos-pygeos-3485ffb0db54f055a5379946dcc31984...,diff --git a/CHANGELOG.rst b/CHANGELOG.rst\nin...
3,cec21ae5f11d9038a7d6cb5ba4fa6f4424eb9b3c,2021-06-09T08:41:42+02:00,TST: rename head branch for GEOS from 'master'...,6,5,2,.github/workflows/test-linux.yml\nci/install_g...,From cec21ae5f11d9038a7d6cb5ba4fa6f4424eb9b3c ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,3fa59c7c52519ee1eb10f9ba7e49ced306201e9f,pygeos-pygeos-3fa59c7c52519ee1eb10f9ba7e49ced3...,diff --git a/.github/workflows/test-linux.yml ...
4,ddb440b39718300cefc7fb12ce43ee1b719f8942,2021-11-11T20:39:30+01:00,[Done] dwithin for GEOS 3.10.0 (#417)\n\n,123,5,4,CHANGELOG.rst\npygeos/predicates.py\npygeos/te...,From ddb440b39718300cefc7fb12ce43ee1b719f8942 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,67fe0ade96c7dbf0bd37633fc62fd13df269e4e4,pygeos-pygeos-67fe0ade96c7dbf0bd37633fc62fd13d...,diff --git a/CHANGELOG.rst b/CHANGELOG.rst\nin...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26312,b68e542a2b2cab4700cb5bb25863efd10bbd0f63,2023-12-11T10:18:33-05:00,Use Mamba Instead of Conda When Running Benchm...,19,7,2,.github/workflows/benchmarks.yml\nasv.conf.json,From b68e542a2b2cab4700cb5bb25863efd10bbd0f63 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/tardis-sn/tardis/...,2025-10-21T23:37:05Z,https://api.github.com/repos/tardis-sn/tardis,public,225,225,False,799e35ba7f333c15b9ab451dac3a287f4d73f4d5,tardis-sn-tardis-799e35ba7f333c15b9ab451dac3a2...,diff --git a/.github/workflows/benchmarks.yml ...
26313,17b1da429ee99c98fd3ae3f140ab7378be769223,2024-07-12T10:18:45-04:00,Refactor and add more benchmarks for montecarl...,305,378,12,.mailmap\nbenchmarks/benchmark_base.py\nbenchm...,From 17b1da429ee99c98fd3ae3f140ab7378be769223 ...,True,| File | Lines Added | Lines Removed | Tot

In [5]:
LIMIT_PER_REPO = 1
def make_task(row):
    owner, repo = row['repo_name'].split('/')
    t = Task(
        owner=owner,
        repo=repo,
        sha=row['pr_base_sha'],
        # python_version=row['analysis_python_version'],
        # env_payload=json.dumps({"dependencies": row['analysis_final_dependencies'].tolist()}),
        tag='pkg',
        # commit_date=pd.to_datetime(row['commit_date']).to_pydatetime(), # <--- replace with real columns.
    )
    return t if t not in registry else None

commits_df['task'] = commits_df.apply(make_task, axis=1)
final_tasks = commits_df.dropna(subset=['task']).groupby('repo_name').head(LIMIT_PER_REPO).reset_index(drop=True)['task'].tolist()


In [6]:
from datasmith.execution.resolution.task_utils import resolve_task

IDX = 12
task = final_tasks[IDX]
# task = Task(owner="pytroll", repo="satpy", sha="0836c021069b7b385c5ff6723779aaf4e66926aa", tag="pkg")
# apache-arrow-nanoarrow-11e73a8c85b45e3d49c8c541b4e1497a649fe03c
task = Task(owner="apache", repo="arrow-nanoarrow", sha="11e73a8c85b45e3d49c8c541b4e1497a649fe03c", tag="pkg")

analysis, task = resolve_task(task, bypass_cache=True)
print(task)
run_labels = gen_run_labels(task, runid=uuid.uuid4().hex)


04:28:09 INFO     datasmith: agent_build_and_validate: task analysis: python_versions=3.12, final_dependencies=['iniconfig==2.0.0', 'numpy==1.26.4', 'packaging==24.0', 'pluggy==1.4.0', 'pyarrow==15.0.2', 'pytest==8.1.1', 'python-dateutil==2.9.0.post0', 'six==1.16.0']


Task(owner='apache', repo='arrow-nanoarrow', sha='11e73a8c85b45e3d49c8c541b4e1497a649fe03c', commit_date=0.0, env_payload='{"dependencies": ["iniconfig==2.0.0", "numpy==1.26.4", "packaging==24.0", "pluggy==1.4.0", "pyarrow==15.0.2", "pytest==8.1.1", "python-dateutil==2.9.0.post0", "six==1.16.0"]}', python_version='3.12', tag='pkg')


## Build + Validate with your script

In [7]:
similar_ctx = registry.get_similar(task)
len(similar_ctx)

0

In [8]:
# Create a DockerContext from your building_data
building_data = Path("scratch/manual_docker_build_pkg.sh").read_text()
ctx = DockerContext(building_data=building_data)
# SIMILAR_IDX=0
# ctx = DockerContext(building_data=similar_ctx[SIMILAR_IDX][1].building_data)


# Build & validate the 'run' image; validator will run profile then tests
res = validator.build_and_validate(
    task=task.with_tag('run'),
    context=ctx,
    run_labels=run_labels,
    build_once_fn=build_once_with_context,
)

print('ok:', res.ok, 'rc:', res.rc)
print('duration_s:', res.duration_s)

print(res.stdout_tail)

04:28:10 INFO     datasmith.docker.validation: build_and_validate[apache-arrow-nanoarrow-11e73a8c85b45e3d49c8c541b4e1497a649fe03c:run]: building image
04:28:10 INFO     datasmith.docker.context: Docker image 'apache-arrow-nanoarrow-11e73a8c85b45e3d49c8c541b4e1497a649fe03c:run' not found locally. Building.
04:28:10 INFO     datasmith.docker.context: $ docker build -t apache-arrow-nanoarrow-11e73a8c85b45e3d49c8c541b4e1497a649fe03c:run . --build-arg REPO_URL='https://www.github.com/apache/arrow-nanoarrow' --build-arg COMMIT_SHA='11e73a8c85b45e3d49c8c541b4e1497a649fe03c' --build-arg ENV_PAYLOAD='{"dependencies": ["iniconfig==2.0.0", "numpy==1.26.4", "packaging==24.0", "pluggy==1.4.0", "pyarrow==15.0.2", "pytest==8.1.1", "python-dateutil==2.9.0.post0", "six==1.16.0"]}' --build-arg PY_VERSION='3.12' --build-arg BASE_IMAGE='buildpack-deps:jammy'


04:28:10 ERROR    datasmith.docker.context: Build failed for 'apache-arrow-nanoarrow-11e73a8c85b45e3d49c8c541b4e1497a649fe03c:run' in 0.1 sec: [unable to find image "sha256:2d73864c2591f7eaf0dccb5b9223c279549f2e58bc2dc2e2e849939e4f37a913"][int.sh /entrypoint.sh
 ---> Using cache
 ---> 18f0f26bc54f
Step 22/43 : RUN chmod +x /entrypoint.sh
]
04:28:10 INFO     datasmith.agents.build: build_once_with_context: result ok=False rc=1 duration=0.1s (stderr_tail_len=95, stdout_tail_len=2320)


ok: False rc: 1
duration_s: 0.06144094467163086
Step 1/43 : ARG BASE_IMAGE=buildpack-deps:jammy
Step 2/43 : ARG PY_VERSION=""                     # passed at build time, used inside stages
Step 3/43 : FROM ${BASE_IMAGE} AS base
 ---> 7e929bacd45f
Step 4/43 : ARG PY_VERSION=""
 ---> Using cache
 ---> 7f0458959f9b
Step 5/43 : RUN apt-get update &&     apt-get install -y --no-install-recommends         jq cmake ninja-build libopenmpi-dev libgeos-dev &&     rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> ae6c0a2d73f7
Step 6/43 : RUN curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest       | tar -xvj -C /usr/local/bin --strip-components=1 bin/micromamba
 ---> Using cache
 ---> 9174d62fdaba
Step 7/43 : ENV MAMBA_ROOT_PREFIX=/opt/conda     PATH=/opt/conda/bin:$PATH     MAMBA_DOCKERFILE_ACTIVATE=1     OPENBLAS_NUM_THREADS=1     MKL_NUM_THREADS=1     OMP_NUM_THREADS=1
 ---> Using cache
 ---> 7371389c942f
Step 8/43 : RUN micromamba install -y -p $MAMBA_ROOT_PREFIX -c conda-forge 

In [9]:
print(res.stderr_tail)

unable to find image "sha256:2d73864c2591f7eaf0dccb5b9223c279549f2e58bc2dc2e2e849939e4f37a913"



In [16]:
print(res.stderr_tail)

=== BUILD ===
(no stderr)

=== PROFILE VALIDATION ===
Using Python 3.11.13 environment at: /opt/conda/envs/asv_3.1
... [truncated] ...
29620/work)
 + pyyaml==6.0.3
 - virtualenv==20.19.0
 + virtualenv==20.35.4
tar: Removing leading `/' from member names
tar: Removing leading `/' from hard link targets


=== TEST VALIDATION ===
+ cd /workspace/repo
+ set +ux
+ '[' 0 -gt 0 ']'
+ '[' -n dbda7d73fbc93e1226c0e8f5bbc46e87ce678052 ']'
+ FORMULACODE_BASE_COMMIT=dbda7d73fbc93e1226c0e8f5bbc46e87ce678052
+ reset_repo_state dbda7d73fbc93e1226c0e8f5bbc46e87ce678052
+ local COMMIT_SHA=dbda7d73fbc93e1226c0e8f5bbc46e87ce678052
++ git remote -v
++ grep '(fetch)'
++ awk '{print $2}'
+ URL=https://www.github.com/pyapp-kit/psygnal
+ [[ https://www.github.com/pyapp-kit/psygnal =~ ^(https://)?(www\.)?github\.com/dask/dask(\.git)?$ ]]
+ [[ https://www.github.com/pyapp-kit/psygnal =~ ^(https://)?(www\.)?github\.com/dask/distributed(\.git)?$ ]]
+ [[ https://www.github.com/pyapp-kit/psygnal =~ ^(https://)?(www\

## Register on success

In [69]:
if res.ok:
    _handle_success(
        ctx=ctx,
        task=task,
        context_registry=registry,
        client=client,
        args=argparse.Namespace(
            context_registry=CONTEXT_REGISTRY_PATH,
            output_dir=OUTPUT_DIR,
            push_to_ecr=False, # Do this manually at the end. Slow.
        )

    )

22:04:25 INFO     datasmith.docker.context: Context registry saved to scratch/merged_context_registry_2025-10-08T09:22:23.904388.json
22:04:25 INFO     datasmith.agents.build: Saved DockerContext pickle: scverse-spatialdata-cab8353e7549a04b2a2538990e0afc2935e54f3f-final.pkl


## Optional: cleanup containers for this run

In [ ]:
# Remove labeled containers; images are left intact
run_id = run_labels.get('datasmith.run', 'unknown')
remove_containers_by_label(client, run_id)
print('Cleaned up containers for run_id:', run_id)
